In [19]:
#Imports

from __future__ import annotations

import json
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union

import osmnx as ox

pd.set_option('display.max_columns', None)  

In [20]:
#Load paths

# Sorties
output_dir = Path("../../Data/input/networkGG")
output_dir.mkdir(parents=True, exist_ok=True)

# Zone d'étude
area_mode = "shapefile"  # "shapefile" | "geocode"
shapefile_path = Path("../../Data/input/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp")
places = ["Genève, Switzerland", "Vandoeuvres, Switzerland", "Cologny, Switzerland", "Collonge-Bellerive, Switzerland", "Choulex, Switzerland"]  # si geocode

# Exports
export_nodes = True
export_edges = True
export_graphml = True
export_formats = ["parquet", "geojson"]  # "parquet" | "geojson" | "gpkg"

# CRS · séparer opérations métriques et export final
operation_crs = "EPSG:2056"  # CRS métrique pour calculs (distances, buffers, etc.)
export_crs = "EPSG:4326"     # CRS final pour les exports

bike_edges_gdf_graph = gpd.read_file(output_dir / "bike_edges_graph.geojson")
bike_nodes_gdf = gpd.read_file(output_dir / "bike_nodes_graph.geojson")

walk_edges_gdf_graph = gpd.read_file(output_dir / "walk_edges_graph.geojson")
walk_nodes_gdf = gpd.read_file(output_dir / "walk_nodes_graph.geojson")


conflict_zones = gpd.read_file(output_dir / "pedestrian_bike_conflicts.geojson")


KeyboardInterrupt: 

## Fonctions utilitaires (simples et réutilisables)

In [ ]:
def load_aoi(area_mode: str, shapefile_path: Path, places: list[str]) -> gpd.GeoDataFrame:
    """Retourne une GeoDataFrame (EPSG:4326) contenant un polygone d'AOI."""
    if area_mode == "shapefile":
        gdf = gpd.read_file(shapefile_path)
        gdf = gdf.to_crs("EPSG:4326") if gdf.crs is not None else gdf.set_crs("EPSG:4326")
        geom = unary_union(gdf.geometry.values)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    if area_mode == "geocode":
        polys = []
        for p in places:
            g = ox.geocode_to_gdf(p).to_crs("EPSG:4326")
            polys.append(unary_union(g.geometry.values))
        geom = unary_union(polys)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    raise ValueError(f"area_mode inconnu: {area_mode}")


def to_crs(gdf: gpd.GeoDataFrame, crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS spécifié en gérant les cas sans CRS défini."""
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326").to_crs(crs)
    return gdf.to_crs(crs)


def to_metric_crs(gdf: gpd.GeoDataFrame, metric_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers un CRS métrique pour les calculs de distance."""
    return to_crs(gdf, metric_crs)


def to_export_crs(gdf: gpd.GeoDataFrame, export_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS d'export final."""
    return to_crs(gdf, export_crs)


def explode_lines(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf[gdf.geometry.notna()].copy()
    return gdf.explode(index_parts=False, ignore_index=True)


def simplify_gdf(gdf: gpd.GeoDataFrame, tol_m: float, metric_crs: str) -> gpd.GeoDataFrame:
    """Simplifie en métrique puis reprojette dans le CRS d'origine."""
    if tol_m and tol_m > 0:
        original_crs = gdf.crs
        gdf_metric = to_metric_crs(gdf, metric_crs)
        gdf_metric = gdf_metric.copy()
        gdf_metric["geometry"] = gdf_metric.geometry.simplify(tol_m, preserve_topology=True)
        return to_crs(gdf_metric, str(original_crs))
    return gdf

import math
import numpy as np
from shapely.geometry import LineString

def mean_bearing(line: LineString) -> float:
    """
    Calcule une orientation moyenne simple d'une LineString (en radians).
    Approche volontairement robuste et peu coûteuse.
    """
    if line is None or line.is_empty:
        return np.nan

    coords = list(line.coords)
    if len(coords) < 2:
        return np.nan

    x0, y0 = coords[0]
    x1, y1 = coords[-1]

    dx = x1 - x0
    dy = y1 - y0

    if dx == 0 and dy == 0:
        return np.nan

    return math.atan2(dy, dx)


def safe_object_columns_for_parquet(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Normalise les colonnes object mixtes (ex. osmid liste/scalar) pour éviter les erreurs parquet."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        if out[col].dtype == "object":
            has_list = out[col].apply(lambda v: isinstance(v, (list, tuple, set))).any()
            if has_list:
                out[col] = out[col].apply(
                    lambda v: None if pd.isna(v) else json.dumps(list(v)) if isinstance(v, (list,tuple,set)) else json.dumps([v])
                )
            else:
                out[col] = out[col].apply(lambda v: None if pd.isna(v) else str(v))
    return out

In [ ]:
aoi = load_aoi(area_mode, shapefile_path, places)  # ta fonction existe déjà
aoi_poly = aoi.geometry.iloc[0]

aoi = load_aoi(area_mode, shapefile_path, places)  # ta fonction existe déjà
aoi_poly = aoi.geometry.iloc[0]


## 2. Get features

1. POI
2. Amenités
3. Edges (déjà présents, simple filtre)
4. Nodes (déjà présents, simple filtre)

In [ ]:
MIN_COLS = ["geometry", "osmid"]

def extract_osm_features(aoi_poly, tags: dict, geom_types=("Point",), keep_cols=None):
    keep_cols = keep_cols or []
    try:
        gdf = ox.features_from_polygon(aoi_poly, tags).reset_index()
    except Exception as e:
        print("OSM extract error:", e)
        return gpd.GeoDataFrame(columns=MIN_COLS + keep_cols, geometry="geometry", crs="EPSG:4326")

    if gdf.empty:
        return gpd.GeoDataFrame(columns=MIN_COLS + keep_cols, geometry="geometry", crs="EPSG:4326")

    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[gdf.geometry.type.isin(list(geom_types))].copy()

    # ---- Assurer une colonne osmid ----
    if "osmid" not in gdf.columns:
        if "id" in gdf.columns:
            gdf["osmid"] = gdf["id"]
        elif "osm_id" in gdf.columns:
            gdf["osmid"] = gdf["osm_id"]
        elif "@id" in gdf.columns:
            gdf["osmid"] = gdf["@id"]
        elif "index" in gdf.columns:
            gdf["osmid"] = gdf["index"]
        else:
            # fallback: créer un id interne (moins idéal mais évite de planter)
            gdf["osmid"] = range(len(gdf))

    cols = ["geometry", "osmid"] + [c for c in keep_cols if c in gdf.columns]
    gdf = gdf[cols].copy()

    gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf



In [ ]:
from pathlib import Path

def export_layer_gpkg(gdf, gpkg_path: Path, layer: str, export_crs="EPSG:4326"):
    if gdf is None or len(gdf) == 0:
        return
    gdf.to_crs(export_crs).to_file(gpkg_path, layer=layer, driver="GPKG")


In [ ]:
POI_VELO_DEFINITIONS = {
    "borne_reparation": {
        "tags": {"amenity": "bicycle_repair_station"},
        "keep_cols": ["amenity", "access"]
    },
    "stationnement_velo": {
        "tags": {"amenity": "bicycle_parking"},
        "keep_cols": ["amenity", "access", "capacity", "covered", "bicycle_parking"]
    },
    "location": {
        "tags": {"amenity": "bicycle_rental"},
        "keep_cols": ["amenity", "operator"]
    }
}

In [ ]:
POI_AMENITES_DEFINITIONS = {
    "amenites": {
        "tags": {
            "amenity": [
                "cafe", "restaurant", "fast_food",
                "pharmacy", "hospital", "clinic",
                "school", "college", "university",
                "kindergarten", "library"
            ],
            "shop": [
                "supermarket", "convenience", "bakery"
            ]
        },
        "keep_cols": ["amenity", "shop", "name"]
    }
}


In [ ]:
EDGE_ATTRIBUTES = {
    "piste": {
        "filter": lambda be: (
            (be.get("cycleway") == "track") |
            (be.get("cycleway:left") == "track") |
            (be.get("cycleway:right") == "track")
        ),
        "keep_cols": ["highway", "cycleway", "cycleway:left", "cycleway:right"]
    },
    "bande": {
        "filter": lambda be: (
            (be.get("cycleway") == "lane") |
            (be.get("cycleway:left") == "lane") |
            (be.get("cycleway:right") == "lane") |
            (be.get("cycleway") == "shared_lane")
        ),
        "keep_cols": ["highway", "cycleway", "cycleway:left", "cycleway:right"]
    },
    "largeur": {
        "filter": lambda be: be.get("width").notna() | be.get("cycleway:width").notna(),
        "keep_cols": ["width", "cycleway:width"]
    },
    "revetement": {
        "filter": lambda be: be.get("surface").notna(),
        "keep_cols": ["surface"]
    },
    "etat_chaussee": {
        "filter": lambda be: be.get("smoothness").notna(),
        "keep_cols": ["smoothness"]
    },
    "eclairage": {
        "filter": lambda be: be.get("lit").notna(),
        "keep_cols": ["lit"]
    },
    "zone_apaisee": {
        "filter": lambda be: (
            (be.get("highway") == "living_street") |
            (be.get("traffic_calming").notna()) |
            (be.get("maxspeed").notna())  # on garde ceux qui ont maxspeed, et tu feras le seuil ensuite si tu veux
        ),
        "keep_cols": ["highway", "maxspeed", "traffic_calming"]
    },
    "sens_inverse": {
        "filter": lambda be: (
            (be.get("oneway").astype(str).isin(["yes", "1", "true"])) &
            (
                be.get("cycleway").astype(str).isin(["opposite_lane", "opposite_track"]) |
                be.get("oneway:bicycle").astype(str).isin(["no", "false", "0"]) |
                be.get("bicycle:backward").astype(str).isin(["yes", "designated"])
            )
        ),
        "keep_cols": ["oneway", "cycleway", "oneway:bicycle", "bicycle:backward"]
    },
    "hauteur_rebord": {
        "filter": lambda be: be.get("kerb").notna() | be.get("kerb:height").notna(),
        "keep_cols": ["kerb", "kerb:height"]
    },
    "topographie": {
        "filter": lambda be: be.get("incline").notna(),
        "keep_cols": ["incline"]
    },
}


In [ ]:
NODE_ATTRIBUTES = {
    "crossing": {
        "filter": lambda bn: bn.get("crossing").notna(),
        "keep_cols": ["crossing"]
    },
    "traffic_signals": {
        "filter": lambda bn: bn.get("traffic_signals").notna(),
        "keep_cols": ["traffic_signals"]
    },
    "barrier": {
        "filter": lambda bn: bn.get("barrier").notna(),
        "keep_cols": ["barrier"]
    },
    "traffic_calming": {
        "filter": lambda bn: bn.get("traffic_calming").notna(),
        "keep_cols": ["traffic_calming"]
    },
}


In [ ]:
def keep_columns(gdf, cols):
    cols = ["geometry", "osmid"] + [c for c in cols if c in gdf.columns]
    cols = list(dict.fromkeys(cols))  # unique
    return gdf[cols].copy()


In [ ]:
gpkg_attr = output_dir / "osm_attributes.gpkg"

# --- POI vélo + POI amenités
ALL_POI = {}
ALL_POI.update(POI_VELO_DEFINITIONS)
ALL_POI.update(POI_AMENITES_DEFINITIONS)

for layer, cfg in ALL_POI.items():
    gdf = extract_osm_features(aoi_poly, cfg["tags"], geom_types=("Point",), keep_cols=cfg["keep_cols"])
    if not gdf.empty:
        export_layer_gpkg(keep_columns(gdf, cfg["keep_cols"]), gpkg_attr, layer, export_crs)
    print(layer, len(gdf))

# --- Edges
be = bike_edges_gdf_graph.copy()
for layer, cfg in EDGE_ATTRIBUTES.items():
    mask = cfg["filter"](be)
    gdf = be[mask].copy()
    if not gdf.empty:
        export_layer_gpkg(keep_columns(gdf, cfg["keep_cols"]), gpkg_attr, layer, export_crs)
    print(layer, len(gdf))

# --- Nodes (ex sur bike_nodes)
bn = bike_nodes_gdf.copy()
for layer, cfg in NODE_ATTRIBUTES.items():
    mask = cfg["filter"](bn)
    gdf = bn[mask].copy()
    if not gdf.empty:
        export_layer_gpkg(keep_columns(gdf, cfg["keep_cols"]), gpkg_attr, layer, export_crs)
    print(layer, len(gdf))

# --- Conflits (déjà calculé)
if conflict_zones is not None and not conflict_zones.empty:
    export_layer_gpkg(conflict_zones, gpkg_attr, "conflicts_md", export_crs)


borne_reparation 34
stationnement_velo 1632
location 51


KeyboardInterrupt: 